In [1]:
import pandas as pd

df = pd.read_csv("data/combined.csv")
SEASON_ORDER = sorted(df.season.unique())  # e.g. '2019/20' < '2020/21' < ... (sorts correctly as strings here)

# which seasons does each region code appear in?
seasons_by_region = df.groupby("region").season.apply(lambda s: sorted(set(s)))
seasons_by_region

region
AT-07-01       [2019/20, 2020/21, 2021/22, 2022/23, 2023/24, ...
AT-07-02                             [2019/20, 2020/21, 2021/22]
AT-07-02-01                 [2022/23, 2023/24, 2024/25, 2025/26]
AT-07-02-02                 [2022/23, 2023/24, 2024/25, 2025/26]
AT-07-03       [2019/20, 2020/21, 2021/22, 2022/23, 2023/24, ...
AT-07-04                             [2019/20, 2020/21, 2021/22]
AT-07-04-01                 [2022/23, 2023/24, 2024/25, 2025/26]
AT-07-04-02                 [2022/23, 2023/24, 2024/25, 2025/26]
AT-07-05       [2019/20, 2020/21, 2021/22, 2022/23, 2023/24, ...
AT-07-06       [2019/20, 2020/21, 2021/22, 2022/23, 2023/24, ...
AT-07-07       [2019/20, 2020/21, 2021/22, 2022/23, 2023/24, ...
AT-07-08       [2019/20, 2020/21, 2021/22, 2022/23, 2023/24, ...
AT-07-09       [2019/20, 2020/21, 2021/22, 2022/23, 2023/24, ...
AT-07-10       [2019/20, 2020/21, 2021/22, 2022/23, 2023/24, ...
AT-07-11       [2019/20, 2020/21, 2021/22, 2022/23, 2023/24, ...
AT-07-12       [20

In [2]:
# a region is "stable" if it's present in every season from its first appearance to the last overall season.
# anything else either (a) never re-appears after some season (possibly replaced by sub-regions),
# or (b) only starts appearing partway through (possibly a new sub-region).
last_overall_season = SEASON_ORDER[-1]

stopped = {}   # region -> last season it was seen in (and it wasn't the last overall season)
started_late = {}  # region -> first season it was seen in (and it wasn't the first overall season)
for region, seasons in seasons_by_region.items():
    if seasons[-1] != last_overall_season:
        stopped[region] = seasons[-1]
    if seasons[0] != SEASON_ORDER[0]:
        started_late[region] = seasons[0]

print("regions that stop appearing (region -> last season seen):")
for r, s in sorted(stopped.items()):
    print(f"  {r:15s} last seen {s}")
print()
print("regions that only start appearing later (region -> first season seen):")
for r, s in sorted(started_late.items()):
    print(f"  {r:15s} first seen {s}")

regions that stop appearing (region -> last season seen):
  AT-07-02        last seen 2021/22
  AT-07-04        last seen 2021/22
  AT-07-14        last seen 2021/22
  AT-07-17        last seen 2021/22
  AT-07-23        last seen 2024/25
  AT-07-26        last seen 2024/25
  AT-07-29        last seen 2024/25

regions that only start appearing later (region -> first season seen):
  AT-07-02-01     first seen 2022/23
  AT-07-02-02     first seen 2022/23
  AT-07-04-01     first seen 2022/23
  AT-07-04-02     first seen 2022/23
  AT-07-14-01     first seen 2022/23
  AT-07-14-02     first seen 2022/23
  AT-07-14-03     first seen 2022/23
  AT-07-14-04     first seen 2022/23
  AT-07-14-05     first seen 2022/23
  AT-07-17-01     first seen 2022/23
  AT-07-17-02     first seen 2022/23
  AT-07-23-01     first seen 2025/26
  AT-07-23-02     first seen 2025/26
  AT-07-26-01     first seen 2025/26
  AT-07-26-02     first seen 2025/26
  AT-07-29-01     first seen 2025/26
  AT-07-29-02     first se

In [3]:
# match each "stopped" region to the sub-region codes (region + "-01", "-02", ...) that
# start appearing in the very next season -> that's a split, not just a coincidence.
all_regions = set(seasons_by_region.index)

splits = []
for old_region, last_season in stopped.items():
    next_season = SEASON_ORDER[SEASON_ORDER.index(last_season) + 1]
    children = sorted(r for r in all_regions
                       if r.startswith(old_region + "-") and seasons_by_region[r][0] == next_season)
    if children:
        splits.append((old_region, last_season, next_season, children))

print(f"{'old region':15s} {'last season':12s} -> {'first season':12s} new region(s)")
for old_region, last_season, next_season, children in splits:
    print(f"{old_region:15s} {last_season:12s} -> {next_season:12s} {', '.join(children)}")

old region      last season  -> first season new region(s)
AT-07-02        2021/22      -> 2022/23      AT-07-02-01, AT-07-02-02
AT-07-04        2021/22      -> 2022/23      AT-07-04-01, AT-07-04-02
AT-07-14        2021/22      -> 2022/23      AT-07-14-01, AT-07-14-02, AT-07-14-03, AT-07-14-04, AT-07-14-05
AT-07-17        2021/22      -> 2022/23      AT-07-17-01, AT-07-17-02
AT-07-23        2024/25      -> 2025/26      AT-07-23-01, AT-07-23-02
AT-07-26        2024/25      -> 2025/26      AT-07-26-01, AT-07-26-02
AT-07-29        2024/25      -> 2025/26      AT-07-29-01, AT-07-29-02, AT-07-29-03


In [4]:
# anything left unexplained? (stopped/started-late regions that aren't part of a clean split above)
explained_old = {old for old, *_ in splits}
explained_new = {c for *_, children in splits for c in children}

unexplained_stopped = set(stopped) - explained_old
unexplained_started = set(started_late) - explained_new
print("stopped but no matching split found:", sorted(unexplained_stopped) or "none")
print("started late but no matching split found:", sorted(unexplained_started) or "none")

stopped but no matching split found: none
started late but no matching split found: none
